# Download YuNet Face Detection Model

Download the YuNet face detection model for use in the face detection module.

**License:** Apache-2.0 (Free for commercial use)

**Model Info:**
- Size: ~340KB
- Format: ONNX
- Input: 320x320 or original size
- Output: Bounding box + 5 landmarks + confidence

## 1. Download Model

In [ ]:
import urllib.request
import os

# Create models directory
os.makedirs('models', exist_ok=True)

# YuNet model URL
model_url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"

# Download
model_path = "models/face_detection_yunet.onnx"

print(f"Downloading YuNet model...")
urllib.request.urlretrieve(model_url, model_path)

# Verify
file_size = os.path.getsize(model_path) / 1024
print(f"Model downloaded: {model_path}")
print(f"File size: {file_size:.1f} KB")

## 2. Test Model

In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# Load model
model_path = "models/face_detection_yunet.onnx"

# Create face detector
detector = cv2.FaceDetectorYN_create(
    model=model_path,
    config="",
    input_size=(320, 320),
    score_threshold=0.7,
    nms_threshold=0.3,
    top_k=5000
)

print("Model loaded successfully!")

In [ ]:
# Create test image with face
image = np.ones((480, 640, 3), dtype=np.uint8) * 240

# Draw face ellipse
cv2.ellipse(image, (320, 240), (100, 120), 0, 0, 360, (180, 160, 140), -1)

# Draw eyes
cv2.circle(image, (280, 220), 15, (60, 60, 60), -1)
cv2.circle(image, (360, 220), 15, (60, 60, 60), -1)

# Draw nose
cv2.ellipse(image, (320, 270), (12, 18), 0, 0, 360, (100, 80, 60), -1)

# Draw mouth
cv2.ellipse(image, (320, 320), (40, 15), 0, 0, 180, (120, 60, 60), 2)

# Display
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Test Image")
plt.axis('off')
plt.show()

In [ ]:
# Detect faces
h, w = image.shape[:2]
detector.setInputSize((w, h))

_, faces = detector.detect(image)

print(f"Detected {len(faces) if faces is not None else 0} face(s)")

if faces is not None:
    for i, face in enumerate(faces):
        print(f"\nFace {i+1}:")
        x, y, w, h = face[:4].astype(int)
        conf = face[14]
        print(f"  Bbox: ({x}, {y}, {w}, {h})")
        print(f"  Confidence: {conf:.3f}")

In [ ]:
# Visualize detection
vis_image = image.copy()

if faces is not None:
    for face in faces:
        x, y, w, h = face[:4].astype(int)
        conf = face[14]
        
        # Draw bounding box
        cv2.rectangle(vis_image, (x, y), (x+w, y+h), (0, 255, 0), 2)
        
        # Draw confidence
        cv2.putText(vis_image, f"{conf:.2f}", (x, y-10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        # Draw landmarks
        landmarks = face[4:14].reshape(5, 2).astype(int)
        colors = [(0, 0, 255), (0, 255, 0), (255, 0, 0), (0, 255, 255), (255, 0, 255)]
        for (lx, ly), color in zip(landmarks, colors):
            cv2.circle(vis_image, (int(lx), int(ly)), 3, color, -1)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(vis_image, cv2.COLOR_BGR2RGB))
plt.title("Face Detection Result")
plt.axis('off')
plt.show()

## 3. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy model to Drive
!mkdir -p /content/drive/MyDrive/models/retin-verify
!cp models/face_detection_yunet.onnx /content/drive/MyDrive/models/retin-verify/

print("Model saved to Google Drive!")

## 4. Download Instructions

### Option 1: Direct Download
```bash
wget -O models/face_detection_yunet.onnx \
  https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx
```

### Option 2: From Google Drive
1. Download from your Google Drive
2. Place in `models/` directory

### Option 3: Use Haar Cascade (Fallback)
If YuNet model is not available, the system will automatically use Haar Cascade:
```python
from face_detection import FaceDetector

# Will use Haar if YuNet not found
detector = FaceDetector()
```

## Model Specifications

| Property | Value |
|----------|-------|
| Model | YuNet (2023 March) |
| Framework | ONNX |
| Size | ~340 KB |
| Input | 320x320 or dynamic |
| Output | Bbox + 5 landmarks + confidence |
| License | Apache-2.0 |

### Output Format
```
[x, y, w, h, x_re, y_re, x_le, y_le, x_nt, y_nt, x_rcm, y_rcm, x_lcm, y_lcm, confidence]
```

Where:
- `(x, y, w, h)`: Bounding box
- `(x_re, y_re)`: Right eye
- `(x_le, y_le)`: Left eye
- `(x_nt, y_nt)`: Nose tip
- `(x_rcm, y_rcm)`: Right mouth corner
- `(x_lcm, y_lcm)`: Left mouth corner